# Consultas SQL · cuaderno del proyecto
**Estud-IA · Análisis de Datos Básico**

Corre las celdas en orden con el botón ▶ (o Shift + Enter). Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

1. La celda 1 prepara todo. No hay que cambiar nada.
2. La celda 2 carga tu muestra.
3. La celda 3 cuenta las filas: tu primera **cifra de control**.
4. Desde la sesión 3: tus dos consultas, más abajo.

In [ ]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


## 2. Carga tu muestra

**Opción A (recomendada).** En GitHub abre tu CSV, copia la dirección de la página (o la del botón **Raw**) y pégala entre las comillas de `ENLACE`.

**Opción B.** Deja `ENLACE = ""` y corre la celda: aparece un botón para subir el archivo desde tu computador.

Con el enlace que viene (el caso de la papelería) deben salir **8 filas y 6 columnas**.

In [ ]:
# Celda 2 · Cargar la muestra
ENLACE = "https://github.com/stevenjm97/Equipo-2-An-lisis-de-datos-B-sico/blob/main/datos/originales/eventos_sintetico.xlsx"

if ENLACE:
    datos = cargar(ENLACE)
else:
    from google.colab import files
    subido = files.upload()
    datos = cargar(next(iter(subido)))

print(f"Cargué {len(datos)} filas y {len(datos.columns)} columnas.")
datos.head()

Cargué 609 filas y 10 columnas.


,evento_id,fecha,tipo_evento,sector_cliente,ciudad,asistentes,precio_venta,costo_estimado,costo_real,estado
0,EV001,2024-01-01 00:00:00,Conferencia,Educación,Medellín,110,21140000,13290000,13250000.0,realizado
1,EV002,2024-01-02 00:00:00,Fiesta corporativa,Tecnología,Bogotá,427,70540000,49700000,57230000.0,realizado
2,EV003,2024-01-07 00:00:00,Lanzamiento de producto,Comercio,Cali,236,49010000,31040000,31280000.0,realizado
3,EV004,2024-01-08 00:00:00,Feria comercial,Comercio,Bogotá,451,42930000,31010000,33020000.0,realizado
4,EV005,2024-01-10 00:00:00,Feria comercial,Comercio,Cali,479,45010000,31710000,34680000.0,realizado


## 3. ¿Cuántas filas hay?

Compara con lo que escribiste en el diccionario, en «Filas en la muestra». Si no coincide, anótalo en la bitácora.

In [ ]:
# Celda 3 · Contar filas
sql("""
SELECT COUNT(*) AS filas
FROM datos
""")

,filas
0,609


## 4. Los nombres exactos de tus columnas

Cópialos de aquí. Si un nombre tiene espacios, tildes o mayúsculas, escríbelo entre **comillas dobles**: `"Fecha del hecho"`. Los valores de texto van entre **comillas simples**: `'completada'`.

In [ ]:
# Celda 4 · Columnas de la muestra
list(datos.columns)
"evento_id"
"fecha"
"tipo_evento"
"sector_cliente"
"ciudad"
"asistentes"
"precio_venta"
"costo_estimado"
"costo_real"
"estado"

'estado'

---
# Sesión 3 · Tus dos consultas

**Consulta 1 · Mirar y filtrar.** Primero cuenta todas las filas; después cuenta las que deja tu filtro; al final mira algunas. Cambia `estado`, `'completada'` y las columnas por las de tu muestra.

Con la papelería: 8 filas en total y 7 después del filtro.

In [ ]:
# Celda 5 · Consulta 1a: filas antes del filtro
sql("""
SELECT COUNT(*) AS filas_total
FROM datos
""")

,filas_total
0,609


Se cuentan con eveventos se han cotizado

In [ ]:
# Celda 6 · Consulta 1b: filas después del filtro
sql("""
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'realizado'
""")

,filas_filtradas
0,575


In [ ]:
# Celda 7 · Consulta 1c: mirar las filas filtradas, solo las columnas que necesitas
sql("""
SELECT evento_id, tipo_evento, precio_venta, sector_cliente
FROM datos
WHERE estado = 'realizado'
ORDER BY precio_venta DESC
LIMIT 10
""")

,evento_id,tipo_evento,precio_venta,sector_cliente
0,EV103,Feria comercial,75810000,Comercio
1,EV115,Fiesta corporativa,74200000,Financiero
2,EV008,Fiesta corporativa,71770000,Tecnología
3,EV355,Fiesta corporativa,71470000,Financiero
4,EV199,Fiesta corporativa,71030000,Comercio
5,EV002,Fiesta corporativa,70540000,Tecnología
6,EV331,Fiesta corporativa,69400000,Financiero
7,EV592,Fiesta corporativa,68660000,Financiero
8,EV200,Feria comercial,68000000,Salud
9,EV392,Fiesta corporativa,66940000,Tecnología


**Consulta 2 · Resumir.** Cuenta cuántas filas hay en cada grupo. Cambia `estado` por la columna con la que quieres comparar (comuna, mes, categoría…).

**Cifra de control:** la suma de los grupos debe dar lo mismo que el total de la celda 5. Con la papelería: completada 7 + cancelada 1 = 8.

In [ ]:
# Celda 8 · Consulta 2: resumen por grupo
sql("""
SELECT
    UPPER(tipo_evento) AS tipo_evento_unificado,
    SUM(precio_venta) AS ingresos_totales,
    COUNT(*) AS filas
FROM datos
GROUP BY tipo_evento_unificado
ORDER BY filas DESC
LIMIT 10
""")

,tipo_evento_unificado,ingresos_totales,filas
0,CAPACITACIÓN,1.818090e+09,176
1,CONFERENCIA,4.657230e+09,156
2,FIESTA CORPORATIVA,4.508390e+09,103
3,LANZAMIENTO DE PRODUCTO,3.729610e+09,87
4,FERIA COMERCIAL,3.894730e+09,87


### Consulta adicional: Eventos con mayor margen de rentabilidad
Calculamos la rentabilidad restando el costo real al precio de venta (`precio_venta - costo_real`) y ordenamos los resultados para ver cuáles fueron los eventos más rentables.

In [ ]:
sql("""
SELECT
    evento_id,
    tipo_evento,
    sector_cliente,
    ciudad,
    precio_venta,
    costo_real,
    (precio_venta - costo_real) AS margen_rentabilidad
FROM datos
WHERE estado = 'realizado'
ORDER BY margen_rentabilidad DESC
LIMIT 10
""")

,evento_id,tipo_evento,sector_cliente,ciudad,precio_venta,costo_real,margen_rentabilidad
0,EV563,Lanzamiento de producto,Salud,Bogotá,62820000,40210000.0,22610000.0
1,EV075,Lanzamiento de producto,Comercio,Medellín,62870000,40340000.0,22530000.0
2,EV473,Lanzamiento de producto,Comercio,Cali,63220000,40850000.0,22370000.0
3,EV171,Lanzamiento de producto,Salud,Bogotá,64530000,43110000.0,21420000.0
4,EV446,Lanzamiento de producto,Salud,Bogotá,60740000,40370000.0,20370000.0
5,EV422,Lanzamiento de producto,Comercio,Medellín,61920000,41780000.0,20140000.0
6,EV015,Lanzamiento de producto,Comercio,Medellín,54640000,35280000.0,19360000.0
7,EV500,Conferencia,Tecnología,Medellín,46700000,27410000.0,19290000.0
8,EV156,Lanzamiento de producto,Tecnología,Medellín,62200000,43080000.0,19120000.0
9,EV598,Lanzamiento de producto,Salud,Medellín,55840000,36780000.0,19060000.0


### Consulta adicional: Ingresos y rentabilidad por sector de cliente
Analizamos el rendimiento financiero agrupando los eventos por el sector del cliente para identificar cuáles son los más lucrativos.

In [ ]:
sql("""
SELECT
    sector_cliente,
    COUNT(*) AS total_eventos,
    SUM(precio_venta) AS ingresos_totales,
    SUM(precio_venta - costo_real) AS rentabilidad_total,
    ROUND(AVG((precio_venta - costo_real) / precio_venta) * 100, 2) AS margen_promedio_porcentaje
FROM datos
WHERE estado = 'realizado'
GROUP BY sector_cliente
ORDER BY rentabilidad_total DESC
""")

,sector_cliente,total_eventos,ingresos_totales,rentabilidad_total,margen_promedio_porcentaje
0,Tecnología,126,4.843320e+09,1.309650e+09,28.84
1,Comercio,123,4.396660e+09,1.018570e+09,27.05
2,Salud,107,2.789420e+09,8.615000e+08,34.99
3,Financiero,109,3.507460e+09,8.331000e+08,28.17
4,Educación,110,1.816610e+09,7.227500e+08,40.63


### Consulta adicional: Eventos que superaron el presupuesto
Identificamos los eventos donde el `costo_real` fue mayor que el `costo_estimado`, calculando el monto exacto del sobrecosto para ayudar a ajustar futuras estimaciones.

In [ ]:
sql("""
SELECT
    evento_id,
    tipo_evento,
    sector_cliente,
    ciudad,
    costo_estimado,
    costo_real,
    (costo_real - costo_estimado) AS sobrecosto
FROM datos
WHERE estado = 'realizado' AND costo_real > costo_estimado
ORDER BY sobrecosto DESC
LIMIT 15
""")

,evento_id,tipo_evento,sector_cliente,ciudad,costo_estimado,costo_real,sobrecosto
0,EV355,Fiesta corporativa,Financiero,Medellín,48820000,62080000.0,13260000.0
1,EV592,Fiesta corporativa,Financiero,Medellín,48120000,61000000.0,12880000.0
2,EV200,Feria comercial,Salud,Bogotá,48190000,60210000.0,12020000.0
3,EV199,Fiesta corporativa,Comercio,Medellín,50610000,62130000.0,11520000.0
4,EV387,Feria comercial,Comercio,Medellín,45460000,56690000.0,11230000.0
5,EV305,Fiesta corporativa,Financiero,Bogotá,41470000,52500000.0,11030000.0
6,EV538,Fiesta corporativa,Tecnología,Bogotá,40260000,51190000.0,10930000.0
7,EV381,Fiesta corporativa,Tecnología,Bogotá,41490000,52380000.0,10890000.0
8,EV430,Fiesta corporativa,Tecnología,Bogotá,45280000,56110000.0,10830000.0
9,EV237,Fiesta corporativa,Financiero,Medellín,43990000,54020000.0,10030000.0


In [ ]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
sql("""
SELECT SUM(filas) AS suma_de_los_grupos
FROM (
    SELECT estado, COUNT(*) AS filas
    FROM datos
    GROUP BY estado

)
""")

,suma_de_los_grupos
0,609.0


In [ ]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
sql("""
SELECT estado, COUNT(*) AS filas
FROM datos
GROUP BY estado
ORDER BY filas DESC
""")

,estado,filas
0,realizado,575
1,cancelado,34


## Guarda tus consultas en GitHub

En tu repositorio, entra a `consultas/`, **Add file › Create new file**, ponle nombre (`01_mirar.sql`, `02_resumen.sql`), pega la consulta con tres comentarios arriba y haz commit con un mensaje que diga qué averiguaste:

```sql
-- Pregunta: ¿cuántas ventas se completaron?
-- Datos: datos/originales/ventas.csv (8 filas)
-- Cifra de control: 8 en total, 7 completadas
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'completada';
```

Mira el ejemplo resuelto en `consultas/` del repositorio de la papelería.

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.